In [ ]:
# Phase 2: 时间标准化与特征工程

**目的**: 统一时间戳格式，提取5G语义特征，为图构建做准备

**输入**: Phase 1分析结果和原始数据

**输出**: 标准化时间戳和5G语义特征


In [1]:
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime, timedelta

print("Phase 2: 时间标准化与特征工程环境准备完成")


Phase 2: 时间标准化与特征工程环境准备完成


In [ ]:
## 2.1 时间戳标准化函数


In [2]:
def standardize_timestamps(df, base_timestamp=1663200000):
    """将MM:SS.s格式转换为Unix时间戳"""
    
    def parse_time_string(time_str):
        if pd.isna(time_str) or time_str == '':
            return np.nan
        
        # 解析MM:SS.s格式
        time_str = str(time_str)
        if ':' in time_str:
            parts = time_str.split(':')
            if len(parts) == 2:
                minutes = int(parts[0])
                seconds_parts = parts[1].split('.')
                seconds = int(seconds_parts[0])
                microseconds = int(seconds_parts[1]) if len(seconds_parts) > 1 else 0
                
                total_seconds = minutes * 60 + seconds + microseconds / 10
                return base_timestamp + total_seconds
        
        return base_timestamp + float(time_str)
    
    # 转换时间戳
    df['StartTime_unix'] = df['StartTime'].apply(parse_time_string)
    df['LastTime_unix'] = df['LastTime'].apply(parse_time_string)
    df['Duration'] = df['LastTime_unix'] - df['StartTime_unix']
    
    return df

print("时间戳标准化函数定义完成")


时间戳标准化函数定义完成


In [ ]:
# 从Phase 1加载清洗后的数据
print("从Phase 1加载清洗后的数据...")
df = pd.read_parquet('../data/processed/clean_data.parquet')
print(f"加载数据完成: {len(df)} 条记录")

def extract_5g_semantic_features(df):
    """提取5G语义特征"""
    
    # 协议类型映射
    protocol_mapping = {
        'tcp': 'TCP',
        'udp': 'UDP', 
        'icmp': 'ICMP',
        'sctp': 'SCTP',  # 5G信令协议
        'arp': 'ARP'
    }
    
    # 标准化协议名称
    df['Proto_normalized'] = df['Proto'].str.lower().map(protocol_mapping).fillna(df['Proto'])
    
    # 5G控制面识别 (SCTP协议通常用于5G信令)
    df['is_5g_control_plane'] = df['Proto_normalized'] == 'SCTP'
    
    # QoS标记解析 (从sTos和dTos字段)
    df['qos_class_src'] = df['sTos'].fillna('unknown')
    df['qos_class_dst'] = df['dTos'].fillna('unknown')
    
    # 流量方向性特征
    df['traffic_ratio'] = df['SrcBytes'] / (df['DstBytes'] + 1)  # 避免除零
    df['packet_ratio'] = df['SrcPkts'] / (df['DstPkts'] + 1)
    
    # 5G特征标记
    df['has_5g_features'] = (
        df['is_5g_control_plane'] | 
        df['qos_class_src'].str.contains('ef|af|cs', case=False, na=False) |
        df['qos_class_dst'].str.contains('ef|af|cs', case=False, na=False)
    )
    
    return df

# 执行5G语义特征提取
print("\\n提取5G语义特征...")
df = extract_5g_semantic_features(df)
print("5G语义特征提取完成")

# 统计5G特征
print(f"\\n5G控制面流量: {df['is_5g_control_plane'].sum()} 条记录")
print(f"包含5G特征的流量: {df['has_5g_features'].sum()} 条记录")
print(f"SCTP协议记录: {(df['Proto_normalized'] == 'SCTP').sum()} 条")


In [ ]:
# 执行时间标准化
print("\\n执行时间标准化...")
df_processed = standardize_timestamps(df)

# 验证时间转换效果
conversion_success = df_processed['StartTime_unix'].notna().sum()
total_records = len(df_processed)
conversion_rate = conversion_success / total_records

print(f"时间戳转换成功率: {conversion_rate:.2%}")
print(f"成功转换: {conversion_success:,} / {total_records:,} 条记录")

# 显示时间转换示例
print("\\n时间转换示例:")
sample_indices = df_processed['StartTime_unix'].notna()
sample_df = df_processed[sample_indices].head(5)
for i, row in sample_df.iterrows():
    print(f"原始: {row['StartTime']} -> Unix: {row['StartTime_unix']:.2f} -> 持续时间: {row['Duration']:.2f}s")

# 时间分布可视化
plt.figure(figsize=(15, 10))

# 1. 时间戳分布
plt.subplot(2, 3, 1)
valid_start_times = df_processed['StartTime_unix'].dropna()
plt.hist(valid_start_times, bins=50, alpha=0.7)
plt.title('StartTime Unix时间戳分布')
plt.xlabel('Unix Timestamp')

# 2. 持续时间分布
plt.subplot(2, 3, 2)
valid_durations = df_processed['Duration'].dropna()
valid_durations = valid_durations[valid_durations >= 0]  # 过滤负值
plt.hist(valid_durations, bins=50, alpha=0.7)
plt.title('流持续时间分布')
plt.xlabel('Duration (seconds)')

# 3. 5G特征分布
plt.subplot(2, 3, 3)
feature_counts = df_processed['has_5g_features'].value_counts()
plt.pie(feature_counts.values, labels=['无5G特征', '有5G特征'], autopct='%1.1f%%')
plt.title('5G特征分布')

# 4. 协议标准化效果
plt.subplot(2, 3, 4)
proto_counts = df_processed['Proto_normalized'].value_counts()
plt.bar(proto_counts.index, proto_counts.values)
plt.title('标准化协议分布')
plt.xticks(rotation=45)

# 5. QoS类别分布
plt.subplot(2, 3, 5)
qos_counts = df_processed['qos_class_src'].value_counts().head(10)
plt.bar(range(len(qos_counts)), qos_counts.values)
plt.title('QoS类别分布 (Top 10)')
plt.xticks(range(len(qos_counts)), qos_counts.index, rotation=45)

# 6. 流量比例分布
plt.subplot(2, 3, 6)
traffic_ratios = df_processed['traffic_ratio'].dropna()
traffic_ratios = traffic_ratios[traffic_ratios < 100]  # 过滤极值
plt.hist(traffic_ratios, bins=50, alpha=0.7)
plt.title('流量比例分布')
plt.xlabel('Traffic Ratio')

plt.tight_layout()
plt.savefig('../data/processed/metadata/time_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# 保存处理后的数据
print("\\n保存时间标准化后的数据...")
df_processed.to_parquet('../data/processed/time_standardized.parquet', index=False)
print(f"时间标准化数据已保存: {len(df_processed)} 条记录")

# 生成质量报告
time_quality_report = {
    "timestamp": datetime.now().isoformat(),
    "total_records": len(df_processed),
    "time_conversion_rate": float(conversion_rate),
    "successful_conversions": int(conversion_success),
    "5g_control_plane_flows": int(df_processed['is_5g_control_plane'].sum()),
    "flows_with_5g_features": int(df_processed['has_5g_features'].sum()),
    "protocol_distribution": df_processed['Proto_normalized'].value_counts().to_dict(),
    "average_duration": float(valid_durations.mean()) if len(valid_durations) > 0 else 0,
    "duration_std": float(valid_durations.std()) if len(valid_durations) > 0 else 0
}

with open('../data/processed/metadata/time_quality_report.json', 'w', encoding='utf-8') as f:
    json.dump(time_quality_report, f, indent=2, ensure_ascii=False)

print("\\n✓ Phase 2 完成:")
print(f"  - 时间标准化数据: ../data/processed/time_standardized.parquet")
print(f"  - 质量报告: ../data/processed/metadata/time_quality_report.json")
print(f"  - 时间分布图表: ../data/processed/metadata/time_distribution.png")
